# CLARA Clustering Analysis with Bayesian Optimization

This notebook implements CLARA (Clustering Large Applications) using `sklearn_extra.cluster` with:
- **Bayesian Optimization** for hyperparameter tuning (n_clusters, n_sampling_iter)
- **Comprehensive Metrics**: Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index, Inertia/WCSS, Dunn Index
- **Visualizations**: Silhouette Plot, 2D/3D Scatter Plots, Elbow Method, Geographic Map, Cluster Sizes, Medoid Distances, Cluster Heatmap

CLARA is a sampling-based extension of PAM (Partitioning Around Medoids) designed for large datasets. Unlike K-Means which uses centroids (mean of cluster points), CLARA uses medoids (actual data points as cluster centers).

In [1]:
# Imports and setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
from config import FIRES_ENGINEERED, FIRES_MERGED, BALANCED_DATASET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Clustering
from sklearn_extra.cluster import CLARA

# Metrics
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Bayesian Optimization
from skopt import gp_minimize
from skopt.space import Integer
from skopt.utils import use_named_args

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-darkgrid')

# Dunn index function (min inter-cluster distance / max intra-cluster diameter)
def compute_dunn_index(X, labels):
    """Compute Dunn Index: ratio of minimum inter-cluster distance to maximum intra-cluster diameter."""
    unique_clusters = np.unique(labels)
    unique_clusters = unique_clusters[unique_clusters != -1] if len(unique_clusters) > 0 else unique_clusters
    if len(unique_clusters) < 2:
        return np.nan

    # Compute cluster diameters (max pairwise distance within each cluster)
    max_diameter = 0.0
    for c in unique_clusters:
        pts = X[labels == c]
        if len(pts) <= 1:
            continue
        dists = np.linalg.norm(pts[:, None, :] - pts[None, :, :], axis=2)
        diam = dists.max()
        if diam > max_diameter:
            max_diameter = diam

    # Compute min inter-cluster centroid distances
    centroids = []
    for c in unique_clusters:
        pts = X[labels == c]
        if len(pts) == 0:
            continue
        centroids.append(pts.mean(axis=0))
    centroids = np.array(centroids)
    inter_dists = np.linalg.norm(centroids[:, None, :] - centroids[None, :, :], axis=2)
    np.fill_diagonal(inter_dists, np.inf)
    min_inter = inter_dists.min()

    if max_diameter == 0 or np.isinf(min_inter):
        return np.nan
    return min_inter / max_diameter

def compute_wcss(X, labels, medoids):
    """Compute Within-Cluster Sum of Squares (WCSS) / Inertia."""
    wcss = 0.0
    unique_labels = np.unique(labels)
    for i, c in enumerate(unique_labels):
        if c == -1:
            continue
        pts = X[labels == c]
        if len(medoids) > i:
            medoid = medoids[i]
            wcss += np.sum((pts - medoid) ** 2)
    return wcss

print('✅ Imports complete - CLARA from sklearn_extra is available')

✅ Imports complete - CLARA from sklearn_extra is available


## 1. Load and Prepare Data

We'll use the balanced dataset for clustering analysis. The data will be scaled using StandardScaler for fair distance calculations.

In [3]:
# Load balanced dataset
df = pd.read_csv(BALANCED_DATASET)
print(f'Dataset shape: {df.shape}')
print(f'Class distribution:\n{df["class"].value_counts()}')
print(f'\nColumns: {df.columns.tolist()}')

# Store coordinates for geographic visualization
coords = df[['longitude', 'latitude']].copy()

# Prepare features (exclude coordinates and target)
feature_cols = [col for col in df.columns if col not in ['longitude', 'latitude', 'class']]
X_df = df[feature_cols]

# Check for non-numeric columns and remove them
non_numeric_cols = X_df.select_dtypes(include=['object', 'category']).columns.tolist()
if non_numeric_cols:
    print(f'\n⚠️ Dropping non-numeric columns: {non_numeric_cols}')
    X_df = X_df.select_dtypes(include=[np.number])
    feature_cols = X_df.columns.tolist()

# Handle missing values
if X_df.isnull().sum().sum() > 0:
    print(f'⚠️ Filling {X_df.isnull().sum().sum()} missing values with column means')
    X_df = X_df.fillna(X_df.mean())

X = X_df.values
y = df['class'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nFeature matrix shape: {X_scaled.shape}')
print(f'Features used: {len(feature_cols)}')

Dataset shape: (7648, 42)
Class distribution:
class
1    3824
0    3824
Name: count, dtype: int64

Columns: ['longitude', 'latitude', 'class', 'prec_winter', 'tmax_winter', 'tmin_winter', 'prec_spring', 'tmax_spring', 'tmin_spring', 'prec_summer', 'tmax_summer', 'tmin_summer', 'prec_autumn', 'tmax_autumn', 'tmin_autumn', 'elevation', 'slope', 'aspect', 'roughness', 'soil_unit_id', 'COARSE', 'SAND', 'SILT', 'CLAY', 'TEXTURE_USDA', 'TEXTURE_SOTER', 'BULK', 'REF_BULK', 'ORG_CARBON', 'PH_WATER', 'TOTAL_N', 'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'TEB', 'BSAT', 'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND']

⚠️ Dropping non-numeric columns: ['TEXTURE_SOTER']

Feature matrix shape: (7648, 38)
Features used: 38


## 2. Bayesian Optimization for CLARA Hyperparameters

We'll use Bayesian Optimization (Gaussian Process) to find optimal:
- **n_clusters**: Number of clusters (2-10)
- **n_sampling_iter**: Number of sampling iterations for CLARA (1-10)

The objective is to maximize the Silhouette Score.

In [ ]:
# Define hyperparameter search space
space = [
    Integer(2, 10, name='n_clusters'),
    Integer(1, 10, name='n_sampling_iter')
]

# Track optimization history
optimization_history = []

@use_named_args(space)
def objective(n_clusters, n_sampling_iter):
    """Objective function for Bayesian Optimization (minimize negative silhouette)."""
    try:
        clara = CLARA(
            n_clusters=n_clusters,
            n_sampling_iter=n_sampling_iter,
            random_state=42
        )
        labels = clara.fit_predict(X_scaled)
        
        # Compute silhouette score
        if len(np.unique(labels)) < 2:
            return 1.0  # Worst case (we minimize negative silhouette)
        
        sil_score = silhouette_score(X_scaled, labels)
        
        # Store results
        optimization_history.append({
            'n_clusters': n_clusters,
            'n_sampling_iter': n_sampling_iter,
            'silhouette': sil_score
        })
        
        return -sil_score  # Negative because we minimize
    
    except Exception as e:
        print(f'Error with n_clusters={n_clusters}, n_sampling_iter={n_sampling_iter}: {e}')
        return 1.0

# Run Bayesian Optimization
print('🔍 Running Bayesian Optimization for CLARA hyperparameters...')
print('Search space: n_clusters=[2-10], n_sampling_iter=[1-10]')
print('Objective: Maximize Silhouette Score\n')

result = gp_minimize(
    objective,
    space,
    n_calls=30,
    n_random_starts=10,
    random_state=42,
    verbose=True
)

# Best parameters
best_n_clusters = result.x[0]
best_n_sampling_iter = result.x[1]
best_silhouette = -result.fun

print(f'\n✅ Bayesian Optimization Complete!')
print(f'Best n_clusters: {best_n_clusters}')
print(f'Best n_sampling_iter: {best_n_sampling_iter}')
print(f'Best Silhouette Score: {best_silhouette:.4f}')

## 3. Optimization Convergence Plot

Visualize how the Bayesian Optimization explored the hyperparameter space.

In [ ]:
# Plot optimization convergence
from skopt.plots import plot_convergence, plot_objective

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence plot
plot_convergence(result, ax=axes[0])
axes[0].set_title('Bayesian Optimization Convergence')

# Optimization history
df_history = pd.DataFrame(optimization_history)
scatter = axes[1].scatter(df_history['n_clusters'], df_history['n_sampling_iter'], 
                          c=df_history['silhouette'], cmap='viridis', s=100, edgecolors='black')
axes[1].scatter(best_n_clusters, best_n_sampling_iter, c='red', s=200, marker='*', 
                label=f'Best: k={best_n_clusters}, iter={best_n_sampling_iter}', edgecolors='black', linewidths=2)
axes[1].set_xlabel('n_clusters')
axes[1].set_ylabel('n_sampling_iter')
axes[1].set_title('Hyperparameter Search Space')
axes[1].legend()
plt.colorbar(scatter, ax=axes[1], label='Silhouette Score')

plt.tight_layout()
plt.show()

# Show optimization history table
print('\nOptimization History (sorted by silhouette):')
display(df_history.sort_values('silhouette', ascending=False).head(10))

## 4. Fit CLARA with Best Parameters

Train the final CLARA model using the optimized hyperparameters.

In [ ]:
# Fit CLARA with best parameters
clara_best = CLARA(
    n_clusters=best_n_clusters,
    n_sampling_iter=best_n_sampling_iter,
    random_state=42
)
labels = clara_best.fit_predict(X_scaled)
medoids = clara_best.cluster_centers_

print(f'✅ CLARA fitted with k={best_n_clusters}, n_sampling_iter={best_n_sampling_iter}')
print(f'Number of clusters: {len(np.unique(labels))}')
print(f'Medoids shape: {medoids.shape}')
print(f'\nCluster distribution:')
for c in np.unique(labels):
    print(f'  Cluster {c}: {np.sum(labels == c)} samples ({100*np.sum(labels == c)/len(labels):.1f}%)')

## 5. Compute All Clustering Metrics

Calculate comprehensive metrics: Silhouette, Davies-Bouldin, Calinski-Harabasz, WCSS/Inertia, and Dunn Index.

In [ ]:
# Compute all clustering metrics
sil_score = silhouette_score(X_scaled, labels)
dbi_score = davies_bouldin_score(X_scaled, labels)
ch_score = calinski_harabasz_score(X_scaled, labels)
wcss = compute_wcss(X_scaled, labels, medoids)
dunn_score = compute_dunn_index(X_scaled, labels)

# Create metrics summary
metrics_summary = pd.DataFrame({
    'Metric': ['Silhouette Score', 'Davies-Bouldin Index', 'Calinski-Harabasz Index', 
               'WCSS (Inertia)', 'Dunn Index'],
    'Value': [sil_score, dbi_score, ch_score, wcss, dunn_score],
    'Interpretation': [
        'Higher is better (-1 to 1)',
        'Lower is better (≥0)',
        'Higher is better',
        'Lower is better (elbow method)',
        'Higher is better'
    ]
})

print('📊 CLARA Clustering Metrics Summary:')
print('=' * 60)
display(metrics_summary)

print(f'\n🎯 Quick Summary:')
print(f'  • Silhouette Score: {sil_score:.4f} (cluster separation quality)')
print(f'  • Davies-Bouldin Index: {dbi_score:.4f} (lower = better cluster separation)')
print(f'  • Calinski-Harabasz Index: {ch_score:.2f} (higher = denser, well-separated clusters)')
print(f'  • WCSS/Inertia: {wcss:.2f} (within-cluster compactness)')
print(f'  • Dunn Index: {dunn_score:.4f} (higher = better inter/intra cluster ratio)')

## 6. Elbow Method - WCSS vs Number of Clusters

Evaluate WCSS across different k values to identify the optimal number of clusters.

In [ ]:
# Elbow method analysis
k_range = range(2, 11)
elbow_metrics = []

for k in k_range:
    clara_temp = CLARA(n_clusters=k, n_sampling_iter=best_n_sampling_iter, random_state=42)
    temp_labels = clara_temp.fit_predict(X_scaled)
    temp_medoids = clara_temp.cluster_centers_
    
    wcss_k = compute_wcss(X_scaled, temp_labels, temp_medoids)
    sil_k = silhouette_score(X_scaled, temp_labels)
    dbi_k = davies_bouldin_score(X_scaled, temp_labels)
    ch_k = calinski_harabasz_score(X_scaled, temp_labels)
    dunn_k = compute_dunn_index(X_scaled, temp_labels)
    
    elbow_metrics.append({
        'k': k,
        'WCSS': wcss_k,
        'Silhouette': sil_k,
        'DBI': dbi_k,
        'CH': ch_k,
        'Dunn': dunn_k
    })
    print(f'k={k}: WCSS={wcss_k:.2f}, Silhouette={sil_k:.4f}, DBI={dbi_k:.4f}')

df_elbow = pd.DataFrame(elbow_metrics)

# Plot elbow method and metrics
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# WCSS / Elbow Plot
axes[0, 0].plot(df_elbow['k'], df_elbow['WCSS'], 'bo-', linewidth=2, markersize=8)
axes[0, 0].axvline(x=best_n_clusters, color='red', linestyle='--', label=f'Best k={best_n_clusters}')
axes[0, 0].set_xlabel('Number of Clusters (k)')
axes[0, 0].set_ylabel('WCSS (Inertia)')
axes[0, 0].set_title('Elbow Method - WCSS')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Silhouette Score
axes[0, 1].plot(df_elbow['k'], df_elbow['Silhouette'], 'go-', linewidth=2, markersize=8)
axes[0, 1].axvline(x=best_n_clusters, color='red', linestyle='--', label=f'Best k={best_n_clusters}')
axes[0, 1].set_xlabel('Number of Clusters (k)')
axes[0, 1].set_ylabel('Silhouette Score')
axes[0, 1].set_title('Silhouette Score vs k')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Davies-Bouldin Index
axes[0, 2].plot(df_elbow['k'], df_elbow['DBI'], 'ro-', linewidth=2, markersize=8)
axes[0, 2].axvline(x=best_n_clusters, color='blue', linestyle='--', label=f'Best k={best_n_clusters}')
axes[0, 2].set_xlabel('Number of Clusters (k)')
axes[0, 2].set_ylabel('Davies-Bouldin Index')
axes[0, 2].set_title('Davies-Bouldin Index vs k (Lower is Better)')
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# Calinski-Harabasz Index
axes[1, 0].plot(df_elbow['k'], df_elbow['CH'], 'mo-', linewidth=2, markersize=8)
axes[1, 0].axvline(x=best_n_clusters, color='red', linestyle='--', label=f'Best k={best_n_clusters}')
axes[1, 0].set_xlabel('Number of Clusters (k)')
axes[1, 0].set_ylabel('Calinski-Harabasz Index')
axes[1, 0].set_title('Calinski-Harabasz Index vs k')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Dunn Index
axes[1, 1].plot(df_elbow['k'], df_elbow['Dunn'], 'co-', linewidth=2, markersize=8)
axes[1, 1].axvline(x=best_n_clusters, color='red', linestyle='--', label=f'Best k={best_n_clusters}')
axes[1, 1].set_xlabel('Number of Clusters (k)')
axes[1, 1].set_ylabel('Dunn Index')
axes[1, 1].set_title('Dunn Index vs k')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

# Summary table
axes[1, 2].axis('off')
table_data = df_elbow.round(4).values
col_labels = df_elbow.columns.tolist()
table = axes[1, 2].table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
axes[1, 2].set_title('Metrics Summary Table', y=0.95)

plt.tight_layout()
plt.show()

## 7. Silhouette Plot

Visualize the silhouette coefficients for each sample grouped by cluster.

In [ ]:
# Silhouette Plot
sample_silhouette_values = silhouette_samples(X_scaled, labels)

fig, ax = plt.subplots(figsize=(12, 8))

y_lower = 10
colors = plt.cm.tab10(np.linspace(0, 1, best_n_clusters))

for i in range(best_n_clusters):
    # Get silhouette values for samples in cluster i
    cluster_silhouette_values = sample_silhouette_values[labels == i]
    cluster_silhouette_values.sort()
    
    cluster_size = len(cluster_silhouette_values)
    y_upper = y_lower + cluster_size
    
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_values,
                      facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
    
    # Label the cluster
    ax.text(-0.05, y_lower + 0.5 * cluster_size, str(i), fontsize=12, fontweight='bold')
    
    y_lower = y_upper + 10

# Add vertical line for average silhouette score
ax.axvline(x=sil_score, color='red', linestyle='--', linewidth=2, 
           label=f'Average Silhouette: {sil_score:.4f}')

ax.set_xlabel('Silhouette Coefficient', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
ax.set_title(f'Silhouette Plot for CLARA Clustering (k={best_n_clusters})', fontsize=14)
ax.legend(loc='upper right')
ax.set_xlim([-0.1, 1])

plt.tight_layout()
plt.show()

# Per-cluster silhouette statistics
print('\n📊 Per-Cluster Silhouette Statistics:')
for i in range(best_n_clusters):
    cluster_sil = sample_silhouette_values[labels == i]
    print(f'  Cluster {i}: Mean={np.mean(cluster_sil):.4f}, Min={np.min(cluster_sil):.4f}, Max={np.max(cluster_sil):.4f}, Count={len(cluster_sil)}')

## 8. 2D and 3D Scatter Plots with Clusters

Visualize clusters using PCA-reduced dimensions for both 2D and 3D projections.

In [ ]:
# PCA for dimensionality reduction
pca_2d = PCA(n_components=2, random_state=42)
pca_3d = PCA(n_components=3, random_state=42)

X_2d = pca_2d.fit_transform(X_scaled)
X_3d = pca_3d.fit_transform(X_scaled)

# Transform medoids to PCA space
medoids_2d = pca_2d.transform(medoids)
medoids_3d = pca_3d.transform(medoids)

print(f'PCA 2D explained variance: {pca_2d.explained_variance_ratio_.sum():.2%}')
print(f'PCA 3D explained variance: {pca_3d.explained_variance_ratio_.sum():.2%}')

# Create figure with 2D and 3D plots
fig = plt.figure(figsize=(16, 6))

# 2D Scatter Plot
ax1 = fig.add_subplot(121)
scatter1 = ax1.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10', alpha=0.6, s=30)
ax1.scatter(medoids_2d[:, 0], medoids_2d[:, 1], c='black', marker='X', s=200, 
            edgecolors='white', linewidths=2, label='Medoids')
ax1.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
ax1.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
ax1.set_title(f'2D PCA Projection - CLARA Clusters (k={best_n_clusters})')
ax1.legend()
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# 3D Scatter Plot
ax2 = fig.add_subplot(122, projection='3d')
scatter2 = ax2.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], c=labels, cmap='tab10', alpha=0.6, s=30)
ax2.scatter(medoids_3d[:, 0], medoids_3d[:, 1], medoids_3d[:, 2], c='black', marker='X', 
            s=200, edgecolors='white', linewidths=2, label='Medoids')
ax2.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]:.1%})')
ax2.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]:.1%})')
ax2.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]:.1%})')
ax2.set_title(f'3D PCA Projection - CLARA Clusters (k={best_n_clusters})')
ax2.legend()

plt.tight_layout()
plt.show()

## 9. Geographic Map - Cluster Distribution

Visualize the geographic distribution of clusters on a map of Algeria and Tunisia.

In [ ]:
# Geographic Map Visualization
import geopandas as gpd

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Load country boundaries if available
try:
    from config import ALGERIA_BOUNDARY, TUNISIA_BOUNDARY
    algeria_gdf = gpd.read_file(ALGERIA_BOUNDARY)
    tunisia_gdf = gpd.read_file(TUNISIA_BOUNDARY)
    has_boundaries = True
except:
    has_boundaries = False
    print('⚠️ Country boundaries not found, plotting points only')

# Cluster colors
colors = plt.cm.tab10(np.linspace(0, 1, best_n_clusters))

# Map 1: Clusters on geographic map
ax1 = axes[0]
if has_boundaries:
    algeria_gdf.boundary.plot(ax=ax1, color='gray', linewidth=1)
    tunisia_gdf.boundary.plot(ax=ax1, color='gray', linewidth=1)

for i in range(best_n_clusters):
    mask = labels == i
    ax1.scatter(coords.loc[mask, 'longitude'], coords.loc[mask, 'latitude'], 
                c=[colors[i]], s=20, alpha=0.6, label=f'Cluster {i} (n={mask.sum()})')

ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.set_title(f'Geographic Distribution of CLARA Clusters (k={best_n_clusters})')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(alpha=0.3)

# Map 2: Colored by actual class (Fire/Non-Fire)
ax2 = axes[1]
if has_boundaries:
    algeria_gdf.boundary.plot(ax=ax2, color='gray', linewidth=1)
    tunisia_gdf.boundary.plot(ax=ax2, color='gray', linewidth=1)

fire_mask = y == 1
non_fire_mask = y == 0
ax2.scatter(coords.loc[non_fire_mask, 'longitude'], coords.loc[non_fire_mask, 'latitude'], 
            c='blue', s=20, alpha=0.4, label=f'Non-Fire (n={non_fire_mask.sum()})')
ax2.scatter(coords.loc[fire_mask, 'longitude'], coords.loc[fire_mask, 'latitude'], 
            c='red', s=20, alpha=0.6, label=f'Fire (n={fire_mask.sum()})')

ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.set_title('Geographic Distribution by Actual Class')
ax2.legend(loc='upper left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Cluster Sizes Distribution

Visualize the distribution of samples across clusters.

In [ ]:
# Cluster sizes analysis
cluster_sizes = pd.Series(labels).value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Bar plot of cluster sizes
colors = plt.cm.tab10(np.linspace(0, 1, best_n_clusters))
bars = axes[0].bar(cluster_sizes.index, cluster_sizes.values, color=colors, edgecolor='black')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Cluster Sizes')
axes[0].set_xticks(range(best_n_clusters))
for bar, size in zip(bars, cluster_sizes.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
                 str(size), ha='center', fontsize=10)

# Pie chart of cluster proportions
axes[1].pie(cluster_sizes.values, labels=[f'Cluster {i}' for i in cluster_sizes.index], 
            colors=colors, autopct='%1.1f%%', startangle=90, explode=[0.02]*best_n_clusters)
axes[1].set_title('Cluster Proportions')

# Cluster vs Actual Class distribution
cluster_class_dist = pd.crosstab(labels, y, normalize='index') * 100
cluster_class_dist.columns = ['Non-Fire (%)', 'Fire (%)']
cluster_class_dist.plot(kind='bar', ax=axes[2], color=['blue', 'red'], edgecolor='black')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Percentage')
axes[2].set_title('Class Distribution within Each Cluster')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

# Print cluster statistics
print('\n📊 Cluster Statistics:')
print(f'Total samples: {len(labels)}')
print(f'\nCluster sizes:')
for c in range(best_n_clusters):
    fire_in_cluster = np.sum((labels == c) & (y == 1))
    non_fire_in_cluster = np.sum((labels == c) & (y == 0))
    print(f'  Cluster {c}: {cluster_sizes[c]} samples ({100*cluster_sizes[c]/len(labels):.1f}%) '
          f'| Fire: {fire_in_cluster} ({100*fire_in_cluster/cluster_sizes[c]:.1f}%) '
          f'| Non-Fire: {non_fire_in_cluster} ({100*non_fire_in_cluster/cluster_sizes[c]:.1f}%)')

## 11. Medoid Distances Analysis

Analyze distances between medoids and from samples to their assigned medoids.

In [ ]:
# Medoid distances analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Inter-medoid distance matrix
medoid_dist_matrix = np.zeros((best_n_clusters, best_n_clusters))
for i in range(best_n_clusters):
    for j in range(best_n_clusters):
        medoid_dist_matrix[i, j] = np.linalg.norm(medoids[i] - medoids[j])

sns.heatmap(medoid_dist_matrix, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0],
            xticklabels=[f'C{i}' for i in range(best_n_clusters)],
            yticklabels=[f'C{i}' for i in range(best_n_clusters)])
axes[0].set_title('Inter-Medoid Distance Matrix')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Cluster')

# 2. Distance from each sample to its assigned medoid
sample_to_medoid_distances = []
for i, (point, label) in enumerate(zip(X_scaled, labels)):
    dist = np.linalg.norm(point - medoids[label])
    sample_to_medoid_distances.append({'cluster': label, 'distance': dist})

df_distances = pd.DataFrame(sample_to_medoid_distances)

# Box plot of distances per cluster
cluster_order = list(range(best_n_clusters))
colors = plt.cm.tab10(np.linspace(0, 1, best_n_clusters))
bp = axes[1].boxplot([df_distances[df_distances['cluster'] == c]['distance'].values for c in cluster_order],
                      labels=[f'C{c}' for c in cluster_order], patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Distance to Medoid')
axes[1].set_title('Sample-to-Medoid Distances by Cluster')
axes[1].grid(alpha=0.3)

# 3. Distribution of all sample-to-medoid distances
axes[2].hist(df_distances['distance'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[2].axvline(df_distances['distance'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {df_distances["distance"].mean():.3f}')
axes[2].axvline(df_distances['distance'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median: {df_distances["distance"].median():.3f}')
axes[2].set_xlabel('Distance to Medoid')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Sample-to-Medoid Distances')
axes[2].legend()

plt.tight_layout()
plt.show()

# Summary statistics
print('\n📊 Medoid Distance Statistics:')
print(f'\nInter-medoid distances:')
print(f'  Min: {medoid_dist_matrix[np.triu_indices(best_n_clusters, k=1)].min():.4f}')
print(f'  Max: {medoid_dist_matrix[np.triu_indices(best_n_clusters, k=1)].max():.4f}')
print(f'  Mean: {medoid_dist_matrix[np.triu_indices(best_n_clusters, k=1)].mean():.4f}')

print(f'\nSample-to-medoid distances:')
for c in range(best_n_clusters):
    cluster_dists = df_distances[df_distances['cluster'] == c]['distance']
    print(f'  Cluster {c}: Mean={cluster_dists.mean():.4f}, Std={cluster_dists.std():.4f}, Max={cluster_dists.max():.4f}')

## 12. Cluster Feature Heatmap

Visualize the mean feature values for each cluster to understand cluster characteristics.

In [ ]:
# Cluster feature heatmap
df_clustered = df.copy()
df_clustered['cluster'] = labels

# Compute mean of each feature per cluster (using scaled values for comparison)
df_features_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
df_features_scaled['cluster'] = labels

cluster_means = df_features_scaled.groupby('cluster').mean()

# Create heatmap
fig, ax = plt.subplots(figsize=(16, 10))

# Limit to top features if too many
if len(feature_cols) > 20:
    # Select features with highest variance across clusters
    feature_variance = cluster_means.var()
    top_features = feature_variance.nlargest(20).index.tolist()
    cluster_means_plot = cluster_means[top_features]
    title_suffix = ' (Top 20 by Variance)'
else:
    cluster_means_plot = cluster_means
    title_suffix = ''

sns.heatmap(cluster_means_plot.T, annot=True, fmt='.2f', cmap='RdYlBu_r', 
            center=0, ax=ax, linewidths=0.5,
            xticklabels=[f'Cluster {i}' for i in range(best_n_clusters)])

ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Feature (Scaled)', fontsize=12)
ax.set_title(f'Mean Feature Values by Cluster{title_suffix}', fontsize=14)

plt.tight_layout()
plt.show()

# Print most distinctive features per cluster
print('\n📊 Most Distinctive Features per Cluster:')
print('(Features where cluster mean differs most from overall mean)')
overall_mean = df_features_scaled[feature_cols].mean()

for c in range(best_n_clusters):
    cluster_mean = cluster_means.loc[c]
    diff = (cluster_mean - overall_mean).abs()
    top_3 = diff.nlargest(3)
    print(f'\nCluster {c}:')
    for feat, val in top_3.items():
        direction = 'higher' if cluster_mean[feat] > overall_mean[feat] else 'lower'
        print(f'  • {feat}: {direction} than average (diff={val:.3f})')

## 13. Davies-Bouldin Index Analysis

Analyze the Davies-Bouldin Index contribution per cluster pair.

In [ ]:
# Davies-Bouldin Index detailed analysis
# DBI = (1/n) * sum(max(R_ij)) where R_ij = (S_i + S_j) / d(c_i, c_j)
# S_i = average distance of points in cluster i to centroid i

def compute_cluster_scatter(X, labels, medoids):
    """Compute scatter (average distance to medoid) for each cluster."""
    scatters = []
    for i in range(len(medoids)):
        cluster_points = X[labels == i]
        if len(cluster_points) == 0:
            scatters.append(0)
        else:
            distances = np.linalg.norm(cluster_points - medoids[i], axis=1)
            scatters.append(np.mean(distances))
    return np.array(scatters)

scatters = compute_cluster_scatter(X_scaled, labels, medoids)

# Compute R_ij matrix (similarity between clusters)
R_matrix = np.zeros((best_n_clusters, best_n_clusters))
for i in range(best_n_clusters):
    for j in range(best_n_clusters):
        if i != j:
            d_ij = np.linalg.norm(medoids[i] - medoids[j])
            if d_ij > 0:
                R_matrix[i, j] = (scatters[i] + scatters[j]) / d_ij
            else:
                R_matrix[i, j] = np.inf

# DBI contribution per cluster
dbi_per_cluster = np.max(R_matrix, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# R_ij matrix heatmap
sns.heatmap(R_matrix, annot=True, fmt='.3f', cmap='Reds', ax=axes[0],
            xticklabels=[f'C{i}' for i in range(best_n_clusters)],
            yticklabels=[f'C{i}' for i in range(best_n_clusters)])
axes[0].set_title('R_ij Similarity Matrix\n(S_i + S_j) / d(c_i, c_j)')
axes[0].set_xlabel('Cluster j')
axes[0].set_ylabel('Cluster i')

# DBI contribution per cluster
colors = plt.cm.tab10(np.linspace(0, 1, best_n_clusters))
bars = axes[1].bar(range(best_n_clusters), dbi_per_cluster, color=colors, edgecolor='black')
axes[1].axhline(y=dbi_score, color='red', linestyle='--', linewidth=2, label=f'DBI={dbi_score:.4f}')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('max(R_ij)')
axes[1].set_title('DBI Contribution per Cluster')
axes[1].set_xticks(range(best_n_clusters))
axes[1].legend()
for bar, val in zip(bars, dbi_per_cluster):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                 f'{val:.3f}', ha='center', fontsize=9)

# Cluster scatter (compactness)
bars = axes[2].bar(range(best_n_clusters), scatters, color=colors, edgecolor='black')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Scatter (Avg. distance to medoid)')
axes[2].set_title('Cluster Compactness (Lower = Better)')
axes[2].set_xticks(range(best_n_clusters))
for bar, val in zip(bars, scatters):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
                 f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'\n📊 Davies-Bouldin Index Analysis:')
print(f'Overall DBI: {dbi_score:.4f}')
print(f'\nCluster-wise DBI contributions (max R_ij):')
for i in range(best_n_clusters):
    worst_pair = np.argmax(R_matrix[i])
    print(f'  Cluster {i}: {dbi_per_cluster[i]:.4f} (worst similarity with Cluster {worst_pair})')

## 14. Summary and Conclusions

Final summary of CLARA clustering results with all metrics and visualizations.

In [ ]:
# Final Summary
print('=' * 70)
print('                    CLARA CLUSTERING ANALYSIS SUMMARY')
print('=' * 70)

print(f'\n📊 DATASET INFORMATION:')
print(f'  • Total samples: {len(labels)}')
print(f'  • Number of features: {len(feature_cols)}')
print(f'  • Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

print(f'\n⚙️ OPTIMAL PARAMETERS (Bayesian Optimization):')
print(f'  • n_clusters: {best_n_clusters}')
print(f'  • n_sampling_iter: {best_n_sampling_iter}')
print(f'  • Optimization calls: 30')

print(f'\n📈 CLUSTERING METRICS:')
print(f'  ┌─────────────────────────────────┬──────────────┬────────────────────────┐')
print(f'  │ Metric                          │ Value        │ Interpretation         │')
print(f'  ├─────────────────────────────────┼──────────────┼────────────────────────┤')
print(f'  │ Silhouette Score                │ {sil_score:>10.4f}   │ Higher is better       │')
print(f'  │ Davies-Bouldin Index            │ {dbi_score:>10.4f}   │ Lower is better        │')
print(f'  │ Calinski-Harabasz Index         │ {ch_score:>10.2f}   │ Higher is better       │')
print(f'  │ WCSS (Inertia)                  │ {wcss:>10.2f}   │ Lower is better        │')
print(f'  │ Dunn Index                      │ {dunn_score:>10.4f}   │ Higher is better       │')
print(f'  └─────────────────────────────────┴──────────────┴────────────────────────┘')

print(f'\n📊 CLUSTER DISTRIBUTION:')
for c in range(best_n_clusters):
    size = np.sum(labels == c)
    fire_pct = 100 * np.sum((labels == c) & (y == 1)) / size
    print(f'  Cluster {c}: {size:>5} samples ({100*size/len(labels):>5.1f}%) | Fire: {fire_pct:>5.1f}%')

print(f'\n✅ VISUALIZATIONS GENERATED:')
print('  • Bayesian Optimization Convergence')
print('  • Elbow Method (WCSS vs k)')
print('  • Silhouette Plot')
print('  • 2D/3D PCA Scatter Plots')
print('  • Geographic Cluster Distribution Map')
print('  • Cluster Size Distribution')
print('  • Medoid Distance Analysis')
print('  • Cluster Feature Heatmap')
print('  • Davies-Bouldin Index Breakdown')

print('\n' + '=' * 70)
print('                         ANALYSIS COMPLETE')
print('=' * 70)

## 15. Geographic Analysis for k=2, k=3, k=5

This section provides detailed geographic analysis for specific k values:
- **k=2**: Expected North vs South fire risk split
- **k=3**: Regional patterns (coastal, interior, desert)
- **k=5**: More granular geographic clustering

These visualizations help interpret how CLARA clusters map to geographic fire risk zones.

In [ ]:
# =============================================================================
# FIT CLARA FOR k=2, k=3, k=5 AND STORE RESULTS
# =============================================================================
k_values_analysis = [2, 3, 5]
clara_results = {}

print("="*80)
print("GEOGRAPHIC CLUSTERING ANALYSIS - CLARA")
print("="*80)

for k in k_values_analysis:
    print(f"\n{'='*40}")
    print(f"K = {k}")
    print(f"{'='*40}")
    
    # Fit CLARA
    clara_model = CLARA(n_clusters=k, n_sampling_iter=best_n_sampling_iter, random_state=42)
    labels_k = clara_model.fit_predict(X_scaled)
    medoids_k = clara_model.cluster_centers_
    
    # Store results
    clara_results[k] = {
        'labels': labels_k,
        'medoids': medoids_k
    }
    
    # Compute metrics
    ari = adjusted_rand_score(y, labels_k) if len(np.unique(labels_k)) >= 2 else np.nan
    sil = silhouette_score(X_scaled, labels_k) if len(np.unique(labels_k)) >= 2 else np.nan
    
    print(f"ARI: {ari:.4f}")
    print(f"Silhouette: {sil:.4f}")
    
    # Cluster statistics with geographic info
    print(f"\nCluster distribution:")
    for c in range(k):
        mask = labels_k == c
        n_points = mask.sum()
        n_fire = (y[mask] == 1).sum()
        fire_pct = 100 * n_fire / n_points if n_points > 0 else 0
        
        # Geographic info
        lat_mean = coords.loc[mask, 'latitude'].mean()
        lon_mean = coords.loc[mask, 'longitude'].mean()
        lat_min, lat_max = coords.loc[mask, 'latitude'].min(), coords.loc[mask, 'latitude'].max()
        
        print(f"  Cluster {c}: {n_points} pts ({100*n_points/len(labels_k):.1f}%)")
        print(f"    Fire: {n_fire} ({fire_pct:.1f}%) | No-Fire: {n_points - n_fire} ({100-fire_pct:.1f}%)")
        print(f"    Lat: [{lat_min:.2f}, {lat_max:.2f}], mean={lat_mean:.2f}")

print("\n" + "="*80)

### K=2: North vs South Fire Risk Analysis

With k=2, CLARA should identify:
- **Northern cluster**: Mediterranean region with higher vegetation and fire risk
- **Southern cluster**: Saharan region with lower fire occurrence

In [ ]:
# =============================================================================
# K=2 GEOGRAPHIC VISUALIZATION - CLARA
# =============================================================================
k = 2
labels_k2 = clara_results[k]['labels']
coords_np = coords.values  # Convert to numpy for easier indexing

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: Geographic map with clusters
ax1 = axes[0, 0]
colors_k2 = ['#1f77b4', '#ff7f0e']  # Blue, Orange
for c in range(k):
    mask = labels_k2 == c
    ax1.scatter(coords_np[mask, 0], coords_np[mask, 1], 
                c=colors_k2[c], s=15, alpha=0.6, label=f'Cluster {c} (n={mask.sum()})')
ax1.axhline(y=33, color='red', linestyle='--', linewidth=2, label='Lat 33° (N/S boundary)')
ax1.set_xlabel('Longitude', fontsize=12)
ax1.set_ylabel('Latitude', fontsize=12)
ax1.set_title(f'CLARA k={k}: Geographic Cluster Distribution', fontsize=14, fontweight='bold')
ax1.legend(loc='lower left')
ax1.grid(alpha=0.3)

# Plot 2: Actual class distribution
ax2 = axes[0, 1]
fire_mask = y == 1
no_fire_mask = y == 0
ax2.scatter(coords_np[no_fire_mask, 0], coords_np[no_fire_mask, 1], 
            c='blue', s=15, alpha=0.4, label=f'No Fire (n={no_fire_mask.sum()})')
ax2.scatter(coords_np[fire_mask, 0], coords_np[fire_mask, 1], 
            c='red', s=15, alpha=0.6, label=f'Fire (n={fire_mask.sum()})')
ax2.axhline(y=33, color='green', linestyle='--', linewidth=2, label='Lat 33°')
ax2.set_xlabel('Longitude', fontsize=12)
ax2.set_ylabel('Latitude', fontsize=12)
ax2.set_title('Actual Fire/No-Fire Distribution', fontsize=14, fontweight='bold')
ax2.legend(loc='lower left')
ax2.grid(alpha=0.3)

# Plot 3: Class distribution per cluster
ax3 = axes[1, 0]
cluster_class_data = []
for c in range(k):
    mask = labels_k2 == c
    n_fire = (y[mask] == 1).sum()
    n_no_fire = (y[mask] == 0).sum()
    cluster_class_data.append({'Cluster': f'Cluster {c}', 'Fire': n_fire, 'No Fire': n_no_fire})

df_cc = pd.DataFrame(cluster_class_data)
x = np.arange(k)
width = 0.35
bars1 = ax3.bar(x - width/2, df_cc['Fire'], width, label='Fire', color='red', alpha=0.8)
bars2 = ax3.bar(x + width/2, df_cc['No Fire'], width, label='No Fire', color='blue', alpha=0.8)
ax3.set_xlabel('Cluster', fontsize=12)
ax3.set_ylabel('Count', fontsize=12)
ax3.set_title(f'Class Distribution per Cluster (k={k})', fontsize=14, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels([f'Cluster {c}' for c in range(k)])
ax3.legend()
for bar in bars1:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{int(bar.get_height())}', 
             ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{int(bar.get_height())}', 
             ha='center', va='bottom', fontsize=10)

# Plot 4: Latitude distribution per cluster
ax4 = axes[1, 1]
lat_per_cluster = [coords_np[labels_k2 == c, 1] for c in range(k)]
bp = ax4.boxplot(lat_per_cluster, labels=[f'Cluster {c}' for c in range(k)], patch_artist=True)
for patch, color in zip(bp['boxes'], colors_k2):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax4.axhline(y=33, color='red', linestyle='--', linewidth=2, label='Lat 33°')
ax4.set_xlabel('Cluster', fontsize=12)
ax4.set_ylabel('Latitude', fontsize=12)
ax4.set_title('Latitude Distribution per Cluster', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print("\n" + "="*60)
print("CLARA K=2 ANALYSIS SUMMARY")
print("="*60)
for c in range(k):
    mask = labels_k2 == c
    lat_mean = coords_np[mask, 1].mean()
    fire_pct = 100 * (y[mask] == 1).sum() / mask.sum()
    region = "NORTH (Mediterranean)" if lat_mean > 33 else "SOUTH (Sahara)"
    print(f"\nCluster {c} -> {region}")
    print(f"  Mean latitude: {lat_mean:.2f}°")
    print(f"  Fire percentage: {fire_pct:.1f}%")
    print(f"  Interpretation: {'Higher fire risk zone' if fire_pct > 50 else 'Lower fire risk zone'}")

### K=3 and K=5: Regional Pattern Analysis

With more clusters, CLARA reveals finer geographic patterns:
- **k=3**: Coastal, interior, and desert separation
- **k=5**: Sub-regional fire risk zones

In [ ]:
# =============================================================================
# K=3 AND K=5 GEOGRAPHIC VISUALIZATION - CLARA
# =============================================================================

for k in [3, 5]:
    labels_k = clara_results[k]['labels']
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'CLARA k={k}: Geographic and Class Analysis', fontsize=16, fontweight='bold')
    
    # Color palette
    colors = plt.cm.tab10(np.linspace(0, 1, k))
    
    # Plot 1: Geographic map with clusters
    ax1 = axes[0, 0]
    for c in range(k):
        mask = labels_k == c
        ax1.scatter(coords_np[mask, 0], coords_np[mask, 1], 
                    c=[colors[c]], s=15, alpha=0.6, label=f'C{c} (n={mask.sum()})')
    ax1.axhline(y=33, color='black', linestyle='--', linewidth=2, alpha=0.7)
    ax1.set_xlabel('Longitude')
    ax1.set_ylabel('Latitude')
    ax1.set_title('Geographic Cluster Distribution')
    ax1.legend(loc='lower left', fontsize=8)
    ax1.grid(alpha=0.3)
    
    # Plot 2: Clusters with fire overlay
    ax2 = axes[0, 1]
    for c in range(k):
        mask = labels_k == c
        ax2.scatter(coords_np[mask, 0], coords_np[mask, 1], 
                    c=[colors[c]], s=10, alpha=0.3)
    fire_mask = y == 1
    ax2.scatter(coords_np[fire_mask, 0], coords_np[fire_mask, 1], 
                c='red', s=5, alpha=0.8, marker='x', label='Fire locations')
    ax2.set_xlabel('Longitude')
    ax2.set_ylabel('Latitude')
    ax2.set_title('Clusters with Fire Overlay (red X)')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # Plot 3: Fire percentage per cluster
    ax3 = axes[0, 2]
    fire_pcts = []
    for c in range(k):
        mask = labels_k == c
        fire_pct = 100 * (y[mask] == 1).sum() / mask.sum() if mask.sum() > 0 else 0
        fire_pcts.append(fire_pct)
    bars = ax3.bar(range(k), fire_pcts, color=colors, edgecolor='black')
    ax3.axhline(y=50, color='red', linestyle='--', linewidth=2, label='50% threshold')
    ax3.set_xlabel('Cluster')
    ax3.set_ylabel('Fire Percentage (%)')
    ax3.set_title('Fire Risk per Cluster')
    ax3.set_xticks(range(k))
    ax3.legend()
    for bar, pct in zip(bars, fire_pcts):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                 f'{pct:.1f}%', ha='center', fontsize=9)
    
    # Plot 4: Latitude distribution per cluster
    ax4 = axes[1, 0]
    lat_data = [coords_np[labels_k == c, 1] for c in range(k)]
    bp = ax4.boxplot(lat_data, labels=[f'C{c}' for c in range(k)], patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax4.axhline(y=33, color='red', linestyle='--', linewidth=2, label='Lat 33°')
    ax4.set_xlabel('Cluster')
    ax4.set_ylabel('Latitude')
    ax4.set_title('Latitude Distribution')
    ax4.legend()
    
    # Plot 5: Cluster sizes pie chart
    ax5 = axes[1, 1]
    sizes = [np.sum(labels_k == c) for c in range(k)]
    ax5.pie(sizes, labels=[f'C{c}\n({s})' for c, s in enumerate(sizes)], 
            colors=colors, autopct='%1.1f%%', startangle=90)
    ax5.set_title('Cluster Size Distribution')
    
    # Plot 6: PCA 2D visualization
    ax6 = axes[1, 2]
    pca_viz = PCA(n_components=2, random_state=42)
    X_pca_viz = pca_viz.fit_transform(X_scaled)
    for c in range(k):
        mask = labels_k == c
        ax6.scatter(X_pca_viz[mask, 0], X_pca_viz[mask, 1], c=[colors[c]], s=10, alpha=0.5, label=f'C{c}')
    ax6.set_xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]*100:.1f}%)')
    ax6.set_ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]*100:.1f}%)')
    ax6.set_title('PCA Projection')
    ax6.legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"CLARA K={k} CLUSTER SUMMARY")
    print(f"{'='*60}")
    for c in range(k):
        mask = labels_k == c
        lat_mean = coords_np[mask, 1].mean()
        lon_mean = coords_np[mask, 0].mean()
        fire_pct = fire_pcts[c]
        
        # Determine region based on latitude
        if lat_mean > 35:
            region = "COASTAL NORTH"
        elif lat_mean > 33:
            region = "INTERIOR NORTH"
        elif lat_mean > 30:
            region = "TRANSITION ZONE"
        else:
            region = "SAHARA SOUTH"
        
        risk_level = "HIGH" if fire_pct > 60 else "MEDIUM" if fire_pct > 40 else "LOW"
        
        print(f"\nCluster {c}:")
        print(f"  Region: {region} (lat={lat_mean:.1f}°, lon={lon_mean:.1f}°)")
        print(f"  Size: {mask.sum()} points ({100*mask.sum()/len(labels_k):.1f}%)")
        print(f"  Fire risk: {fire_pct:.1f}% -> {risk_level} RISK")

### Metrics Comparison: k=2, k=3, k=5

Summary comparison of clustering quality metrics across different k values.

In [ ]:
# =============================================================================
# METRICS COMPARISON: k=2, k=3, k=5 FOR CLARA
# =============================================================================
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, homogeneity_score, completeness_score

comparison_metrics_clara = []

for k in [2, 3, 5]:
    labels_k = clara_results[k]['labels']
    medoids_k = clara_results[k]['medoids']
    
    # Compute metrics
    ari = adjusted_rand_score(y, labels_k)
    ami = adjusted_mutual_info_score(y, labels_k)
    sil = silhouette_score(X_scaled, labels_k)
    dbi = davies_bouldin_score(X_scaled, labels_k)
    ch = calinski_harabasz_score(X_scaled, labels_k)
    hom = homogeneity_score(y, labels_k)
    comp = completeness_score(y, labels_k)
    wcss_k = compute_wcss(X_scaled, labels_k, medoids_k)
    dunn_k = compute_dunn_index(X_scaled, labels_k)
    
    # Purity
    unique_clusters = np.unique(labels_k)
    cluster_purities = []
    for c in unique_clusters:
        members = y[labels_k == c]
        if len(members) > 0:
            cluster_purities.append(np.bincount(members.astype(int)).max())
    purity = np.sum(cluster_purities) / len(labels_k)
    
    comparison_metrics_clara.append({
        'k': k,
        'ARI': ari,
        'AMI': ami,
        'Silhouette': sil,
        'DBI': dbi,
        'CH': ch,
        'Homogeneity': hom,
        'Completeness': comp,
        'Purity': purity,
        'WCSS': wcss_k,
        'Dunn': dunn_k
    })

df_comp_clara = pd.DataFrame(comparison_metrics_clara)

print("="*80)
print("CLARA METRICS COMPARISON: k=2, k=3, k=5")
print("="*80)
display(df_comp_clara.round(4))

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

metrics_plot = ['ARI', 'AMI', 'Silhouette', 'Purity', 'DBI', 'WCSS']
colors_bar = ['#2ecc71', '#3498db', '#9b59b6']

for i, metric in enumerate(metrics_plot):
    ax = axes[i]
    bars = ax.bar(df_comp_clara['k'].astype(str), df_comp_clara[metric], color=colors_bar, edgecolor='black')
    ax.set_xlabel('k')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs k')
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}' if metric not in ['WCSS', 'CH'] else f'{height:.0f}',
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Recommendations
print("\n" + "="*60)
print("CLARA RECOMMENDATION")
print("="*60)
best_ari_k = df_comp_clara.loc[df_comp_clara['ARI'].idxmax(), 'k']
best_sil_k = df_comp_clara.loc[df_comp_clara['Silhouette'].idxmax(), 'k']
best_dbi_k = df_comp_clara.loc[df_comp_clara['DBI'].idxmin(), 'k']  # Lower is better

print(f"Best k by ARI: k={best_ari_k}")
print(f"Best k by Silhouette: k={best_sil_k}")
print(f"Best k by DBI (lower=better): k={best_dbi_k}")
print(f"\nNote: k=2 shows North/South fire risk split")
print(f"      k=3 or k=5 may reveal more regional fire patterns")

### Interpretation Guidelines (To be filled after running)

After executing the cells above, observe:

**For k=2:**
- [ ] Does one cluster correspond to Northern regions (lat > 33°)?
- [ ] Does the Northern cluster have higher fire percentage?
- [ ] Are medoids positioned in representative locations?

**For k=3:**
- [ ] Do clusters separate coastal, interior, and desert regions?
- [ ] Which cluster has the highest fire risk?
- [ ] How does CLARA's medoid-based clustering compare to K-Means?

**For k=5:**
- [ ] Do clusters reveal sub-regional patterns?
- [ ] Are medoids geographically meaningful?
- [ ] Does CLARA handle outliers better than K-Means?

**CLARA vs K-Means comparison:**
- Compare metrics (ARI, Silhouette) for same k values
- Note if medoid-based clustering gives different geographic patterns
- CLARA should be more robust to outliers than K-Means